# 1 Data collection

samle nyhedsartikler med labels fake / real

typiske datasæt kan f.eks. være:

* FakeNewsNet
* Kaggle fake news datasets
* scraped nyhedsartikler
* de angivet datasæt fra DIKU
* de artikler jeg snakker lidt om under kilder.

målet er at få et datasæt hvor hver artikel har en label. Sådan at vi ved hvad er sandt og falsk. Men vores model ved det ikke og skal selv vurdere. Sådan at vi får nogle samlinige resultater. 


In [28]:
from pathlib import Path
import os

print("Current working directory:", os.getcwd())

raw_path = Path("Data_Opgave_3/news_sample.csv")
splits_dir = Path("Data_Opgave_3/splits")

print("raw_path =", raw_path.resolve())
print("splits_dir =", splits_dir.resolve())

Current working directory: c:\Users\Bruger\VisualCode\GDS_Eksamensprojekt\GDS-Eksamen-FakeNews
raw_path = C:\Users\Bruger\VisualCode\GDS_Eksamensprojekt\GDS-Eksamen-FakeNews\Data_Opgave_3\news_sample.csv
splits_dir = C:\Users\Bruger\VisualCode\GDS_Eksamensprojekt\GDS-Eksamen-FakeNews\Data_Opgave_3\splits


# 2 Data splitting

training bruges til at træne modellen
validation bruges til at vælge hyperparametre
test bruges kun til at måle den endelige performance

Dette er vigtigt for at sikre at modellen kan generalisere til nye data. 

Vi skal have fokus på at der er nogle klassiske problemer som nemt kan opstå. 

**Data leakage:** Problemet opstår når datapunkter ikke er uafhængige.
**Random split:** Den simpleste metode er: shuffle datasættet split i train / validation / test

før modellen trænes deles datasættet i tre dele:

* training data (60–80%)
* validation data (10–20%)
* test data (10–20%)

**Samme data:** Hvis man scraper nyheder kan datasættet indeholde mange artikler fra samme kilde. Ideen er

CNN article
CNN article
CNN article
BBC article
BBC article

Hvis CNN-artikler ligger i både train og test kan modellen lære en bestemt stil i stedet for sandhed.

**Tidsdata:** Nyheder er også tidsafhængige.

Hvis man blander artikler fra 2016–2024 tilfældigt kan modellen lære mønstre fra fremtiden. Men i virkeligheden vil modellen blive brugt på fremtidige artikler.

**Potientiel løsnig:**
Hvis datasættet er lille kan man bruge k-fold cross validation.
Ideen:

* split data i k dele
* træne på k-1 dele
* teste på den sidste
* gentage k gange

Men også her skal man være forsigtig med leakage. Folds skal stadig respektere struktur i data (tid, grupper osv.). 

Formålet er at sikre at modellen evalueres på data der virkelig er ukendt for modellen, så resultatet bedre afspejler hvordan modellen vil fungere i praksis.

In [29]:
import csv
import random
from collections import Counter

def compute_targets(label_counts: Counter, train=0.8, val=0.1, test=0.1):
    targets = {}
    for label, n in label_counts.items():
        n_train = int(round(n * train))
        n_val = int(round(n * val))
        n_test = n - n_train - n_val
        targets[label] = {"train": n_train, "val": n_val, "test": n_test}
    return targets

def choose_split(remaining: dict, rng: random.Random):
    total = remaining["train"] + remaining["val"] + remaining["test"]
    if total <= 0:
        return None
    r = rng.randrange(total)
    if r < remaining["train"]:
        return "train"
    r -= remaining["train"]
    if r < remaining["val"]:
        return "val"
    return "test"

in_path = raw_path
seed = 42

splits_dir.mkdir(parents=True, exist_ok=True)

train_path = splits_dir / "train.csv"
val_path = splits_dir / "val.csv"
test_path = splits_dir / "test.csv"

label_counts = Counter()
with open(in_path, "r", encoding="utf-8", newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        label_counts[row["type"]] += 1

targets = compute_targets(label_counts, train=0.8, val=0.1, test=0.1)
rng = random.Random(seed)
written = Counter()

with open(in_path, "r", encoding="utf-8", newline="") as f_in, \
     open(train_path, "w", encoding="utf-8", newline="") as f_train, \
     open(val_path, "w", encoding="utf-8", newline="") as f_val, \
     open(test_path, "w", encoding="utf-8", newline="") as f_test:

    reader = csv.DictReader(f_in)
    fieldnames = reader.fieldnames

    w_train = csv.DictWriter(f_train, fieldnames=fieldnames)
    w_val = csv.DictWriter(f_val, fieldnames=fieldnames)
    w_test = csv.DictWriter(f_test, fieldnames=fieldnames)

    w_train.writeheader()
    w_val.writeheader()
    w_test.writeheader()

    for row in reader:
        label = row["type"]
        split = choose_split(targets[label], rng)

        if split == "train":
            w_train.writerow(row)
        elif split == "val":
            w_val.writerow(row)
        else:
            w_test.writerow(row)

        targets[label][split] -= 1
        written[split] += 1

print("Wrote:", dict(written))
print("train exists?", train_path.exists())
print("val exists?", val_path.exists())
print("test exists?", test_path.exists())

Wrote: {'val': 26, 'train': 201, 'test': 23}
train exists? True
val exists? True
test exists? True


# 3 Data Preprocessing

teksten renses og standardiseres så modellen kan arbejde med den.

typiske trin er:

* tekst rensning
* lowercase
* fjerne punctuation
* tokenization
* stopwords removal
* stemming eller lemmatization

nogle pipelines erstatter, ting som vi har gjort i tidligere opgaver.

* URL → `<URL>`
* tal → `<NUM>`
* datoer → `<DATE>`

In [30]:
import re
import csv
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

csv.field_size_limit(2**31 - 1)

def clean_text(text):
    text = str(text).lower()

    text = re.sub(
        r"\b(?:jan(?:uary)?|feb(?:ruary)?|mar(?:ch)?|apr(?:il)?|may|jun(?:e)?|jul(?:y)?|aug(?:ust)?|sep(?:t(?:ember)?)?|oct(?:ober)?|nov(?:ember)?|dec(?:ember)?)"
        r"(?!\.\w)"
        r"(?:\.(?=\s|$|\d))?"
        r"(?:\s+(?:0?[1-9]|[12][0-9]|3[01])"
        r"(?:\s*,?\s*(?:\d{2}|\d{4}))?"
        r")?"
        r"(?=\W|$)",
        "<DATE>",
        text
    )

    text = re.sub(r"\b\d+(?:[.,]\d+)?\b", "<NUM>", text)

    text = re.sub(r"https?://\S+", "<URL>", text)
    text = re.sub(r"www\.\S+", "<URL>", text)
    text = re.sub(r"\S+\.com\S*", "<URL>", text)

    text = re.sub(r"\s+", " ", text).strip()
    return text

tokenizer = RegexpTokenizer(r"<URL>|<NUM>|<DATE>|[a-z]+(?:'[a-z]+)?")

def tokenize_nltk(cleaned_text):
    return tokenizer.tokenize(cleaned_text)

try:
    STOPWORDS = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    STOPWORDS = set(stopwords.words("english"))

def remove_stopwords(tokens):
    return [t for t in tokens if t not in STOPWORDS]

stemmer = SnowballStemmer("english")

def stem_tokens(tokens):
    out = []
    for t in tokens:
        if t in ("<URL>", "<NUM>", "<DATE>"):
            out.append(t)
        else:
            out.append(stemmer.stem(t))
    return out

def reduction_rate(before, after):
    if before == 0:
        return 0.0
    return (before - after) / before

def process_file(file_path, out_path, limit_rows=None):
    vocab_raw = set()
    vocab_no_stop = set()
    vocab_stemmed = set()

    total_raw = 0
    total_no_stop = 0
    docs_read = 0

    with open(file_path, "r", encoding="utf-8-sig", newline="") as f_in, \
         open(out_path, "w", encoding="utf-8", newline="") as f_out:

        reader = csv.DictReader(f_in)
        writer = csv.DictWriter(f_out, fieldnames=["type", "processed_content"])
        writer.writeheader()

        for row in reader:
            if limit_rows is not None and docs_read >= limit_rows:
                break

            docs_read += 1

            raw_content = row.get("content", "") or ""
            cleaned = clean_text(raw_content)

            tokens = tokenize_nltk(cleaned)
            tokens_no_stop = remove_stopwords(tokens)
            tokens_stem = stem_tokens(tokens_no_stop)

            total_raw += len(tokens)
            total_no_stop += len(tokens_no_stop)

            vocab_raw.update(tokens)
            vocab_no_stop.update(tokens_no_stop)
            vocab_stemmed.update(tokens_stem)

            writer.writerow({
                "type": row.get("type"),
                "processed_content": " ".join(tokens_stem)
            })

    V_raw = len(vocab_raw)
    V_no_stop = len(vocab_no_stop)
    V_stem = len(vocab_stemmed)

    rr_stop = reduction_rate(V_raw, V_no_stop)
    rr_stem = reduction_rate(V_no_stop, V_stem)
    rr_stem_from_raw = reduction_rate(V_raw, V_stem)
    token_rr_stop = (total_raw - total_no_stop) / total_raw if total_raw else 0

    print(f"\nProcessed file: {file_path.name}")
    print(f"Saved to: {out_path.name}")
    print(f"Sample size (documents): {docs_read:,}")
    print(f"Vocabulary size (tokenized): {V_raw:,}")
    print(f"Vocabulary size (after stopwords): {V_no_stop:,}")
    print(f"Reduction rate (stopwords): {rr_stop:.4f} = {rr_stop*100:.2f}%")
    print(f"Vocabulary size (after stemming): {V_stem:,}")
    print(f"Reduction rate (stemming): {rr_stem:.4f} = {rr_stem*100:.2f}%")
    print(f"Reduction rate (stemming vs raw): {rr_stem_from_raw:.4f} = {rr_stem_from_raw*100:.2f}%")
    print(f"Total tokens (raw): {total_raw:,}")
    print(f"Total tokens (after stopwords): {total_no_stop:,}")
    print(f"Token reduction (stopwords): {token_rr_stop:.4f} = {token_rr_stop*100:.2f}%")

## Kør Data Preprocessing

Vi køre selve koden på vores splits

In [31]:
for split in ["train", "val", "test"]:
    process_file(
        splits_dir / f"{split}.csv",
        splits_dir / f"{split}_processed.csv"
    )

print((splits_dir / "train.csv").exists())
print((splits_dir / "val.csv").exists())
print((splits_dir / "test.csv").exists())


Processed file: train.csv
Saved to: train_processed.csv
Sample size (documents): 201
Vocabulary size (tokenized): 13,633
Vocabulary size (after stopwords): 13,467
Reduction rate (stopwords): 0.0122 = 1.22%
Vocabulary size (after stemming): 8,726
Reduction rate (stemming): 0.3520 = 35.20%
Reduction rate (stemming vs raw): 0.3599 = 35.99%
Total tokens (raw): 137,582
Total tokens (after stopwords): 76,748
Token reduction (stopwords): 0.4422 = 44.22%

Processed file: val.csv
Saved to: val_processed.csv
Sample size (documents): 26
Vocabulary size (tokenized): 3,853
Vocabulary size (after stopwords): 3,717
Reduction rate (stopwords): 0.0353 = 3.53%
Vocabulary size (after stemming): 2,845
Reduction rate (stemming): 0.2346 = 23.46%
Reduction rate (stemming vs raw): 0.2616 = 26.16%
Total tokens (raw): 16,208
Total tokens (after stopwords): 9,191
Token reduction (stopwords): 0.4329 = 43.29%

Processed file: test.csv
Saved to: test_processed.csv
Sample size (documents): 23
Vocabulary size (token

# 4 Feature representation

Først konverterer vi tekst til tal. Dette forstiller jeg mig at vi gør med den mest klassiske metode er:

TF-IDF (Term Frequency (TF) - Inverse Document Frequency (IDF))

Ideen bag er formlen her:

$$
TFIDF(t,d)=TF(t,d)\cdot\log\frac{N}{df(t)}
$$

hvor

* $TF(t,d)$ = hvor ofte ordet forekommer i dokumentet
* $df(t)$ = hvor mange dokumenter ordet findes i
* $N$ = antal dokumenter

ideen er så at vi opdeler ord i forskellige værdier som i dette eksempel:

* ord som “the” → lav værdi
* ord som “conspiracy” → høj værdi

Dette gør at modellen fokuserer på de ord som faktisk giver information om teksten.

Mange artikler viser også at TF-IDF stadig er meget brugt i fake news detection, fordi det er en simpel måde at repræsentere tekst på, men stadig meget effektiv.

Andre typiske metoder er:

* Bag-of-Words
* Word embeddings (Word2Vec / GloVe)
* contextual embeddings (BERT)

klassiske ML-modeller bruger ofte TF-IDF, og er den jeg regner med at bruge. Men BERT virker også effektiv og god. Man kunne overveje mulighederne for at bruge begge hvis det ikke er for krævende.

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

train = pd.read_csv(splits_dir / "train_processed.csv")
val = pd.read_csv(splits_dir / "val_processed.csv")
test = pd.read_csv(splits_dir / "test_processed.csv")

labels = {
    'reliable': 'reliable', 'political': 'reliable',
    'fake': 'fake', 'unreliable': 'fake', 'conspiracy': 'fake',
    'bias': 'fake', 'hate': 'fake', 'clickbait': 'fake'
}

for df in [train, val, test]:
    df["binary_label"] = df["type"].map(labels)

train = train.dropna(subset=["binary_label", "processed_content"])
val = val.dropna(subset=["binary_label", "processed_content"])
test = test.dropna(subset=["binary_label", "processed_content"])


vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2
)

X_train = vectorizer.fit_transform(train["processed_content"])
X_val = vectorizer.transform(val["processed_content"])
X_test = vectorizer.transform(test["processed_content"])

## 5 Feature selection / dimensionality reduction

tekstdata giver meget høje dimensioner (tusindvis af ord).

derfor kan man reducere antallet af features ved:

* feature selection
* chi-square test
* mutual information
* regularization
* dimensionality reduction

Formålet er at beholde informative features og fjerne støj.

In [ ]:
from sklearn.feature_selection import SelectKBest, chi2

selector = SelectKBest(chi2, k=10000)

X_train_sel = selector.fit_transform(X_train, train["binary_label"])
X_val_sel = selector.transform(X_val)
X_test_sel = selector.transform(X_test)

## 6 Model

selve klassifikationsmodellen trænes på de features vi har konstrueret. For at kunne evaluere forskellige metoder kan vi træne flere modeller og sammenligne deres performance.

### Sanity check model

Først trænes en meget simpel model for at sikre at datasættet og feature-repræsentationen fungerer korrekt.
typiske modeller til dette er:

* Naive Bayes

Naive Bayes fungerer ofte overraskende godt til tekstklassifikation fordi modellen antager at ordene optræder uafhængigt af hinanden.
Denne model kan derfor bruges som et sanity check og baseline.

### Klassiske machine learning modeller

derefter kan mere kraftfulde modeller trænes på de samme features.

typiske modeller er:

* Logistic Regression
* Linear SVM

Disse modeller fungerer ofte meget godt sammen med TF-IDF features.

### Deep learning modeller

Noget andet som går igen i mange artikler er brugen af neurale netværk og transformer-modeller. Her lærer modellen selv sproglige repræsentationer direkte fra data. Et eksempel på modeller er:

* BERT
* RoBERTa
* XLNet

Disse modeller kan forstå semantiske og kontekstuelle relationer i tekst, hvilket gør dem meget stærke til opgaver som fake news detection.
I nogle løsninger bruger man også en hybrid model, hvor transformer-modellen først laver en avanceret repræsentation af teksten, og derefter bruges en klassifikationsmodel som f.eks.

* XGBoost
* SVM

til selve klassifikationen.
Dette viser at man kan lave en pipeline hvor:
tekst → embeddings/features → klassifikationsmodel

### Data imbalance

Et problem i fake news detection er ofte **class imbalance**.
Datasættet kan indeholde langt flere rigtige nyheder end falske.

Hvis modellen kun optimerer **accuracy**, kan den ende med at klassificere næsten alt som “real news” og stadig få en høj score.

Derfor bruges ofte **F1-score** til evaluering, fordi den tager højde for både precision og recall.

En anden løsning er at justere selve træningen af modellen ved at bruge **weighted loss**. Her gives større vægt til den klasse der forekommer sjældnere i datasættet. På den måde bliver fejl på fake news straffet mere under træningen.

Ideen kan skrives som

$$
L(y,\hat{y})=-w_k\log(\hat{y}_k)
$$

hvor vægten typisk vælges som

$$
w_k=\frac{1}{|y_k|}
$$

så klasser med færre eksempler får højere vægt.

### Baselines man næsten bør have med: Naive Bayes og “billige” checks
Jeg tænker at vi tester også bruger Naive Bayes som sanity check/baseline. Jeg ser i en del artikler at Naive Bayes, selvom antagelserne er forsimplede, ofte virker godt i praksis til dokumentklassifikation og spamfiltrering, og det er hurtigt og kræver ikke sindssyg meget data. Derfor kan dette bruges som hurtigt check. 

Så i jeg vil have i vores bedste løsning noget i retning af:

Naive Bayes      → sanity check
Logistic / SVM   → stærk baseline
BERT / DL model  → avanceret model

Det gør vores opgave mere robust, fordi vi ikke bare vælger en model uden at benchmarke.


In [ ]:
model.fit(X_train_sel, train["binary_label"])
val_preds = model.predict(X_val_sel)

## 7 Hyperparameter tuning

modellens parametre optimeres via:

* grid search
* random search
* cross-validation

det sker typisk på validation-datasættet.

## 8 Evaluation

Til sidst evalueres modellen på test-data. Her måles hvor godt modellen kan klassificere nye artikler som fake eller real. Da dette er en binær klassifikation, kan hvert resultat placeres i en såkaldt confusion matrix.

Der findes fire mulige udfald:

* **True Positive (TP)** – en fake artikel bliver korrekt klassificeret som fake
* **True Negative (TN)** – en rigtig artikel bliver korrekt klassificeret som real
* **False Positive (FP)** – en rigtig artikel bliver forkert klassificeret som fake
* **False Negative (FN)** – en fake artikel bliver forkert klassificeret som real

Ud fra disse værdier kan man beregne forskellige metrics.

### Accuracy

Accuracy måler hvor stor en andel af alle forudsigelser der er korrekte:

$$
accuracy=\frac{TP+TN}{TP+TN+FP+FN}
$$

Problemet er at accuracy kan være misvisende hvis datasættet er ubalanceret Hvis langt de fleste artikler er “real news”, kan en model få høj accuracy ved næsten altid at forudsige real.

vores forlæser siger at han mener at det ikke er godt at bruge til ubalancerede datasæt. Derfor som udgangspunkt mener han at vi ikke skal bruge det. 

### Precision

Precision måler hvor ofte modellen har ret når den siger at en artikel er fake:

$$
precision=\frac{TP}{TP+FP}
$$

Høj precision betyder få false positiver, altså at modellen sjældent kalder rigtige nyheder for fake.

### Recall

Recall måler hvor stor en andel af alle fake artikler modellen faktisk finder:

$$
recall=\frac{TP}{TP+FN}
$$

Høj recall betyder få falske negativer, altså at modellen sjældent overser fake news.

### Precision–Recall tradeoff

Der findes typisk et tradeoff mellem precision og recall. En model kan få høj recall ved at klassificere mange artikler som fake, men så falder precision fordi flere rigtige artikler bliver markeret forkert.

### F1-score

F1-score kombinerer precision og recall til et tal ved at tage det harmoniske gennemsnit:

$$
F1=2\frac{precision\cdot recall}{precision+recall}
$$

F1-score bruges ofte i fake news detection fordi datasættet kan være ubalanceret, og fordi målet kræver at både precision og recall er høje.

## 9 Deployment / usage 

vi kan teste vores model i praksis eventuelt ved at bruge vores tidligere opgave om at hente artikler fra BBC også teste på dem. Der var over 800 styks. 

* analyserer nye artikler
* klassificerer fake / real
* giver en sandsynlighed for fake news.
